In [1]:
#pip install psycopg2

In [2]:
#pip install SQLAlchemy

In [3]:
from sqlalchemy import create_engine, MetaData, Table
import pandas as pd
from sqlalchemy.orm import sessionmaker

In [4]:
uid = 'postgres'
pwd = 'Postgres%401234'
server = "localhost"
database = 'TrainingDB'

In [5]:
engine = create_engine(f'postgresql://{uid}:{pwd}@{server}:5433/{database}')

In [6]:
sql = "SELECT * From public.person"

In [7]:
df = pd.read_sql_query(sql,engine)

In [8]:
print(df)

   id             name  gender  height_cm           city
0   1         John Doe    Male        180       New York
1   2       Jane Smith  Female        165    Los Angeles
2   3        Sam Brown    Male        175        Chicago
3   4      Emily White  Female        160  San Francisco
4   5  Michael Johnson    Male        185          Miami
5   6     Olivia Davis  Female        170         Boston
6   7     James Miller    Male        178         Dallas


In [9]:
sql2 = "SELECT * From public.person2"

In [10]:
df2 = pd.read_sql_query(sql,engine)

In [11]:
print(df2)

   id             name  gender  height_cm           city
0   1         John Doe    Male        180       New York
1   2       Jane Smith  Female        165    Los Angeles
2   3        Sam Brown    Male        175        Chicago
3   4      Emily White  Female        160  San Francisco
4   5  Michael Johnson    Male        185          Miami
5   6     Olivia Davis  Female        170         Boston
6   7     James Miller    Male        178         Dallas


In [12]:
from flask import Flask, jsonify, request

In [13]:
# Initialize the Flask application
app = Flask(__name__)

In [14]:
Session = sessionmaker(bind=engine)
session = Session()

In [15]:
# Create metadata and bind the tables
metadata = MetaData(bind=engine)

In [16]:
# Assuming you have two tables named 'table1' and 'table2' (adjust as per your schema)
table1 = Table('person', metadata, autoload_with=engine)
table2 = Table('person2', metadata, autoload_with=engine)

In [17]:
# Define the route to get record based on id
@app.route('/get_records/<int:id>', methods=['GET'])
def get_records(id):
    # Fetch data from table1 using SQLAlchemy
    record_from_table1 = session.query(table1).filter(table1.c.id == id).first()
    
    # Fetch data from table2 using SQLAlchemy
    record_from_table2 = session.query(table2).filter(table2.c.id == id).first()
    
    # Prepare the response based on the results from both tables
    response = {}
    
    if record_from_table1:
        response['table1'] = dict(record_from_table1)
    else:
        response['table1'] = None

    if record_from_table2:
        response['table2'] = dict(record_from_table2)
    else:
        response['table2'] = None

    # If no records were found in either table
    if not record_from_table1 and not record_from_table2:
        return jsonify({"message": "Record not found in both tables"}), 404
    
    return jsonify(response), 200

    

In [ ]:
from werkzeug.serving import run_simple

if __name__ == '__main__':
    run_simple('127.0.0.1', 5000, app)

 * Running on http://127.0.0.1:5000/ (Press CTRL+C to quit)
127.0.0.1 - - [21/Jan/2025 12:53:09] "GET /get_records/1 HTTP/1.1" 200 -
127.0.0.1 - - [21/Jan/2025 13:00:47] "GET /get_records/2 HTTP/1.1" 200 -
127.0.0.1 - - [21/Jan/2025 13:13:41] "GET /get_records/8 HTTP/1.1" 404 -
127.0.0.1 - - [21/Jan/2025 13:13:53] "GET /get_records/7 HTTP/1.1" 200 -
127.0.0.1 - - [21/Jan/2025 13:23:59] "GET /get_records/7 HTTP/1.1" 200 -
127.0.0.1 - - [21/Jan/2025 13:24:32] "GET /get_records/8 HTTP/1.1" 404 -
